# HTB Academy — AI Data Attacks: Trojan / Backdoor Study Notebook

This notebook follows the Trojan-attack sections of Module 302. It begins with Section 14's environment, GTSRB dataset setup, and attack configuration. Later attack stages will be added only as their course subsections are reached.

**Current scope: Section 14 only.** We set up reproducible settings and name the source class, target class, and trigger. We do not yet alter any images, change labels, train a model, or test a backdoor.

HTB's lesson downloads its dataset if required files are missing. Here that transfer is controlled by `DOWNLOAD_DATASET`, which is `False` by default. Set it to `True` in the dedicated download cell when you intentionally want the archive in the notebook's working directory. The archive URL is the one linked from the HTB lesson.


## 1. Imports and repeatable environment

PyTorch provides the neural-network library; `torchvision` supplies image transforms and dataset loaders. NumPy and Python's `random` module have separate random-number generators, so all three are seeded. A fixed seed makes many random choices repeatable, although some GPU operations can still be nondeterministic.

Device selection follows the lesson: prefer CUDA when available, otherwise Apple MPS, otherwise CPU. CUDA is NVIDIA's GPU platform; MPS is Apple's Metal backend. The code reports the selected device so we know where later training would run.


In [ ]:
import os
import random
import shutil
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import ImageFolder
from tqdm.auto import tqdm, trange

SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# Ask cuDNN for repeatable algorithms where possible. This may reduce speed;
# a fixed seed alone does not guarantee every GPU calculation is deterministic.
if torch.backends.cudnn.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"PyTorch: {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")
print(f"Using device: {device}")


## 2. GTSRB class IDs and readable names

GTSRB has 43 traffic-sign classes numbered 0 through 42. Models generally work with numeric labels, while people learn more easily from names. This lookup keeps both views connected.

The attack uses **Class 14 (Stop)** as the source and **Class 3 (Speed limit 60 km/h)** as the target. “Source” means which original examples are modified; “target” means the label we want the trigger-bearing examples to teach the model to predict.


In [ ]:
# GTSRB label ID -> human-readable sign name (the course's 43-class mapping).
GTSRB_CLASS_NAMES = {
    0: "Speed limit (20km/h)", 1: "Speed limit (30km/h)",
    2: "Speed limit (50km/h)", 3: "Speed limit (60km/h)",
    4: "Speed limit (70km/h)", 5: "Speed limit (80km/h)",
    6: "End of speed limit (80km/h)", 7: "Speed limit (100km/h)",
    8: "Speed limit (120km/h)", 9: "No passing",
    10: "No passing for vehicles over 3.5 tons",
    11: "Right-of-way at next intersection", 12: "Priority road",
    13: "Yield", 14: "Stop", 15: "No vehicles",
    16: "Vehicles over 3.5 tons prohibited", 17: "No entry",
    18: "General caution", 19: "Dangerous curve left",
    20: "Dangerous curve right", 21: "Double curve", 22: "Bumpy road",
    23: "Slippery road", 24: "Road narrows on the right",
    25: "Road work", 26: "Traffic signals", 27: "Pedestrians",
    28: "Children crossing", 29: "Bicycles crossing",
    30: "Beware of ice/snow", 31: "Wild animals crossing",
    32: "End of speed/pass limits", 33: "Turn right ahead",
    34: "Turn left ahead", 35: "Ahead only", 36: "Go straight or right",
    37: "Go straight or left", 38: "Keep right", 39: "Keep left",
    40: "Roundabout mandatory", 41: "End of no passing",
    42: "End no passing for vehicles over 3.5 tons",
}
NUM_CLASSES_GTSRB = len(GTSRB_CLASS_NAMES)

def get_gtsrb_class_name(class_id):
    """Translate a numeric GTSRB class ID into its readable name."""
    return GTSRB_CLASS_NAMES.get(class_id, f"Unknown Class {class_id}")

print(f"Number of GTSRB classes: {NUM_CLASSES_GTSRB}")
print(f"Source: Class 14 — {get_gtsrb_class_name(14)}")
print(f"Target: Class 3 — {get_gtsrb_class_name(3)}")
assert NUM_CLASSES_GTSRB == 43


## 3. Dataset locations and download helpers

The course expects training images, test images, and a CSV of test labels under `./GTSRB`. Downloads are saved temporarily under `./gtsrb_downloads` and extracted into `./GTSRB`.

`download_file` streams the response in chunks rather than holding the whole archive in memory. `extract_zip` checks that each archive member stays inside the requested extraction directory before writing it. That path check is a small defensive addition to the course helper: archive contents should not be allowed to write outside the chosen dataset folder.

The helper definitions themselves do not access the network. The following cell makes the actual download opt-in, so running all current setup cells will not transfer the dataset by accident.


In [ ]:
# Dataset paths are relative to this notebook's current working directory.
DATASET_ROOT = Path("GTSRB")
DOWNLOAD_DIR = Path("gtsrb_downloads")
DATASET_URL = (
    "https://cdn.services-k8s.prod.aws.htb.systems/content/optional_resources/file/"
    "0c5ae6a1-f8ce-4f54-a505-cde0f9a1b372.zip"
)
TRAIN_DIR = DATASET_ROOT / "Final_Training" / "Images"
TEST_IMAGE_DIR = DATASET_ROOT / "Final_Test" / "Images"
TEST_LABELS_CSV = DATASET_ROOT / "GT-final_test.csv"
DATASET_ZIP_PATH = DOWNLOAD_DIR / "GTSRB.zip"


def download_file(url, destination):
    """Stream a trusted course resource to disk; return its saved path."""
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        print(f"Already downloaded: {destination}")
        return destination

    with requests.get(url, stream=True, timeout=(15, 120)) as response:
        response.raise_for_status()
        with destination.open("wb") as output_file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    output_file.write(chunk)
    return destination


def extract_zip_safely(zip_path, destination):
    """Extract zip members only when their resolved paths stay under destination."""
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            resolved_member = (destination / member.filename).resolve()
            if resolved_member != destination and destination not in resolved_member.parents:
                raise ValueError(f"Unsafe path in archive: {member.filename}")
        archive.extractall(destination)
    return destination


def dataset_files_present():
    """Check the core training and evaluation paths expected by the lesson."""
    return (
        TRAIN_DIR.is_dir()
        and TEST_IMAGE_DIR.is_dir()
        and TEST_LABELS_CSV.is_file()
    )

print(f"Dataset root: {DATASET_ROOT.resolve()}")
print(f"Training images: {TRAIN_DIR}")
print(f"Test images: {TEST_IMAGE_DIR}")
print(f"Test labels: {TEST_LABELS_CSV}")
print(f"Dataset already present: {dataset_files_present()}")


In [ ]:
# Intentionally opt in before downloading the course dataset archive.
DOWNLOAD_DATASET = False

if DOWNLOAD_DATASET:
    if dataset_files_present():
        print("GTSRB data already appears complete; no download needed.")
    else:
        archive_path = download_file(DATASET_URL, DATASET_ZIP_PATH)
        extract_zip_safely(archive_path, DATASET_ROOT)
        print(f"Extraction complete. Expected files present: {dataset_files_present()}")
else:
    print("Dataset download is disabled. Set DOWNLOAD_DATASET = True to fetch the HTB-linked archive.")


## 4. Image preprocessing and Trojan parameters

### Preprocessing values

Images are resized to `48 × 48` RGB pixels. The ImageNet channel means and standard deviations are supplied for normalization. For channel `c`, the common normalization is:

`normalized_pixel = (pixel - mean_c) / std_c`

Read it aloud as **“normalized pixel equals pixel minus the mean for its color channel, divided by that channel's standard deviation.”** This changes the numerical scale the model sees. The magenta trigger values below are defined in the original zero-to-one RGB pixel scale; later code must take care to insert the trigger consistently relative to this normalization.

### Attack settings

The trigger is a 4-by-4 magenta patch near the bottom-right. With a 48-by-48 image, its top-left coordinate is `(43, 43)`: `48 - 4 - 1 = 43`. Since pixel indices run from 0 to 47, the patch covers indices 43–46 and leaves index 47 as a one-pixel margin.

`POISON_RATE = 0.10` means ten percent of source-class examples are intended to be selected. If there are `N_source` Stop images, the basic count idea is `0.10 × N_source`; the exact rounding and selection code comes in a later subsection. The settings here describe the planned attack; they do not yet modify images or labels.


In [ ]:
# Image dimensions and channel-wise normalization constants.
IMG_SIZE = 48
IMG_MEAN = (0.485, 0.456, 0.406)
IMG_STD = (0.229, 0.224, 0.225)

# Trojan goal: triggered Stop examples will be assigned the Speed Limit 60 label.
SOURCE_CLASS = 14
TARGET_CLASS = 3
POISON_RATE = 0.10

# Trigger geometry, measured on the resized 48 x 48 pixel image.
TRIGGER_SIZE = 4
TRIGGER_POS = (
    IMG_SIZE - TRIGGER_SIZE - 1,
    IMG_SIZE - TRIGGER_SIZE - 1,
)
# RGB magenta (red=1, green=0, blue=1), before normalization.
TRIGGER_COLOR_VAL = (1.0, 0.0, 1.0)

# HTB palette used by its later visualizations.
HTB_GREEN = "#9fef00"
NODE_BLACK = "#141d2b"
HACKER_GREY = "#a4b1cd"
WHITE = "#ffffff"
AZURE = "#0086ff"
NUGGET_YELLOW = "#ffaf00"
MALWARE_RED = "#ff3e3e"
VIVID_PURPLE = "#9f00ff"
AQUAMARINE = "#2ee7b6"

# Derive and print the exact position and area to make the configuration tangible.
trigger_top, trigger_left = TRIGGER_POS
trigger_area_pixels = TRIGGER_SIZE * TRIGGER_SIZE
print(f"Image size: {IMG_SIZE} x {IMG_SIZE} RGB")
print(f"Source class: {SOURCE_CLASS} — {get_gtsrb_class_name(SOURCE_CLASS)}")
print(f"Target class: {TARGET_CLASS} — {get_gtsrb_class_name(TARGET_CLASS)}")
print(f"Poison rate: {POISON_RATE:.0%} of source-class examples")
print(f"Trigger: {TRIGGER_SIZE} x {TRIGGER_SIZE} = {trigger_area_pixels} pixel locations")
print(f"Trigger top-left (row, column): {TRIGGER_POS}; RGB: {TRIGGER_COLOR_VAL}")
assert (trigger_top, trigger_left) == (43, 43)
assert trigger_top + TRIGGER_SIZE <= IMG_SIZE - 1
assert trigger_left + TRIGGER_SIZE <= IMG_SIZE - 1
assert SOURCE_CLASS in GTSRB_CLASS_NAMES and TARGET_CLASS in GTSRB_CLASS_NAMES
assert SOURCE_CLASS != TARGET_CLASS


## COAE quick recall — Trojan setup

- **Backdoor/Trojan:** a hidden conditional behavior in the trained model, activated by a trigger.
- **Source class:** genuine examples we modify—in this lesson, Stop (14).
- **Target class:** the label the triggered examples are changed to—Speed limit 60 (3).
- **Poisoning differs from clean-label:** this attack changes both pixels (adds a trigger) and the selected examples' labels; clean-label poisoning retains labels.
- **Trigger:** a 4×4 magenta RGB patch at `(43, 43)` on a 48×48 image, leaving a one-pixel bottom/right margin.
- **Poison rate:** 10% of source-class images are intended to be selected; later code determines the exact count and labels.
- **Current section boundary:** setup and parameters only. No poison examples are made, no model is trained, and no backdoor behavior is tested yet.
- **Study notebook difference:** unlike the HTB code that downloads missing files automatically, this notebook requires setting `DOWNLOAD_DATASET = True` in its clearly labeled cell.

**Exam check:** Explain how ordinary validation images can show good accuracy while failing to reveal a backdoor. Name the trigger, source class, and target class in this example.


## Section 15 — The CNN Model Architecture

This section connects the poisoning idea to the model that will learn it. First, we express ordinary clean-data training and how Trojan examples change its objective. Then we define the GTSRB convolutional neural network (CNN).

**Scope:** architecture definition and a small forward-pass shape check only. No poisoned images are created, the network is not trained, and attack success is not evaluated. Dataset download remains opt-in above.


### Clean supervised-learning objective, read aloud

The model is f(X; W), read **“f of X, parameterized by W.”** X is an input image; W is the collection of learned weights and biases. D_clean is the set of clean image-label pairs.

W* = argmin_W (1 / |D_clean|) × sum over (x_i, y_i) in D_clean of L(f(x_i; W), y_i)

Read it as: **“W star is the value of W that minimizes the average loss across all clean training examples.”** x_i and y_i are **“x sub i”** and **“y sub i”**: image i and its correct class. L is prediction loss. The sum adds each example's loss; dividing by |D_clean| (“the number of clean examples”) makes an average. argmin_W is **“arg-min over W”**: select weights making the expression smallest. W* (“W star”) names that choice.

Small example: losses 0.2, 0.5, and 0.3 average to (0.2 + 0.5 + 0.3) / 3 = 1/3. Training changes W so the average tends to decrease.


### How Trojan samples change the objective

The selected source subset contains Stop signs (class 14). T(x_j), read **“T of x sub j,”** means apply the trigger to image j. A poisoned copy is paired with the target label, Speed limit 60 (class 3).

D_poison = { (T(x_j), y_target) | (x_j, y_source) is in D_source_subset }

Read: **“D poison contains each selected source image after applying the trigger, paired with the target label.”** The vertical bar means “such that.”

D_total = (D_clean set-difference D_source_subset) union D_poison

Read: **“remove the selected original source examples from clean data, then combine the remainder with poisoned examples.”** Set difference removes; union combines. Thus the selected originals are replaced by triggered copies with changed labels.

The modified loss has two parts: loss on the remaining clean examples, plus loss on triggered images paired with y_target. The first teaches ordinary sign recognition; the second teaches the trigger-to-target association. Together this is a dual task. HTB omits normalization details in its displayed formula; the important idea is that both kinds of examples contribute.


### CNN building blocks and dimensions

A convolution is summarized as Y = X * K + b, read **“Y equals X convolved with K, plus b.”** X is an image or feature map; K is a learned filter/kernel sliding over local regions; * means this sliding weighted operation, not ordinary multiplication; b is a learned bias; Y is the feature map output.

ReLU is relu(x) = max(0, x), read **“ReLU of x equals the larger of zero and x.”** For example, [-2, 0.5, 3] becomes [0, 0.5, 3]. ReLU adds nonlinearity. Max pooling keeps the strongest activation in a local neighborhood; a 2×2 pool with stride 2 halves height and width.

Input shape is (batch, 3, 48, 48): batch, RGB channels, rows, columns. A 3×3 convolution with padding 1 preserves spatial size. The architecture goes:
- input: (batch, 3, 48, 48)
- conv1: (batch, 32, 48, 48)
- conv2: (batch, 64, 48, 48)
- pool1: (batch, 64, 24, 24)
- conv3: (batch, 128, 24, 24)
- pool2: (batch, 128, 12, 12)
- flatten: (batch, 18,432), since 128 × 12 × 12 = 18,432
- fc1: (batch, 512)
- fc2: (batch, 43), one raw score/logit per class

Flatten reshapes feature maps into a vector per image so a fully connected layer can combine them. Dropout p=0.5 randomly zeros half the selected activations during training; it is disabled during evaluation. Dropout is regularization, not itself a Trojan defense.


In [ ]:
class GTSRB_CNN(nn.Module):
    # CNN for 48x48 RGB traffic signs and the 43 GTSRB classes.
    def __init__(self, num_classes=NUM_CLASSES_GTSRB):
        super().__init__()
        # 3x3 filter with padding 1 preserves spatial size: 3 RGB -> 32 maps.
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        # Later convolutions combine earlier patterns into more feature maps.
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        # At this point spatial size is 24x24.
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        # 48x48 -> 24x24 -> 12x12; flatten 128*12*12 = 18,432 values.
        self._feature_size = 128 * 12 * 12
        self.fc1 = nn.Linear(self._feature_size, 512)
        self.fc2 = nn.Linear(512, num_classes)
        # During training, each selected activation has a 50% chance of zeroing.
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x):
        # Feature extraction: convolution, ReLU, convolution, ReLU, then pool.
        x = self.pool1(F.relu(self.conv2(F.relu(self.conv1(x)))))
        # Second feature block and spatial down-sampling.
        x = self.pool2(F.relu(self.conv3(x)))
        # Preserve batch axis; combine channels, rows, and columns.
        x = x.view(-1, self._feature_size)
        # Dense classifier: 18,432 features -> 512 -> 43 class logits.
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)


# Construct the model and move its parameters to the selected device.
model_structure_gtsrb = GTSRB_CNN(num_classes=NUM_CLASSES_GTSRB).to(device)
print(f"Feature values per image before dense layers: {model_structure_gtsrb._feature_size:,}")
print(f"Model device: {device}")
print(model_structure_gtsrb)


### Small shape check (no training)

One synthetic all-zero image verifies output shape (1, 43): one image and 43 class logits. No dataset is loaded, no weights are trained, and no Trojan behavior is tested. Batch size 1 keeps this check lightweight.


In [ ]:
# One synthetic image checks the architecture's input/output contract.
# eval() disables dropout; no_grad() avoids storing backpropagation data.
model_structure_gtsrb.eval()
with torch.no_grad():
    sample_image = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=device)
    sample_logits = model_structure_gtsrb(sample_image)
print(f"Input shape: {tuple(sample_image.shape)}")
print(f"Logit shape: {tuple(sample_logits.shape)}")
assert sample_logits.shape == (1, NUM_CLASSES_GTSRB)


### COAE quick recall — CNN and Trojan objective

- argmin_W is the choice of weights W that minimizes loss.
- Clean training minimizes average loss on clean images with correct labels.
- Trojan training adds loss for triggered source images paired with the target label.
- The combined objective preserves normal classification while learning trigger-conditioned behavior.
- Convolution filters produce feature maps; ReLU zeros negative activations; max pooling reduces spatial size.
- The feature path ends at 128×12×12 = 18,432 values, then dense layers map 18,432 → 512 → 43 logits.
- Dropout regularizes training; it does not establish the model is free of a backdoor.
- This section defines and shape-checks the model. Training and attack evaluation come later.

**Exam check:** Explain both parts of the Trojan training objective. Calculate the flattened feature count from 128 maps of size 12×12.
